<a href="https://colab.research.google.com/github/jlee1050/QuantImg/blob/main/Quantum_Image_Manipulation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install qiskit matplotlib 'qiskit[visualization]' Pillow gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 22.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 34.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [qiskit]


In [15]:
import numpy as np
from PIL import Image
from qiskit import QuantumCircuit
from qiskit.quantum_info import Operator

def build_circuit(gates):
    """gates example: [("H", 0), ("X", 1), ("CNOT", 0)]"""
    qc = QuantumCircuit(2)
    for g, q in gates:
        if g == "CNOT":
            qc.cx(0, 1)                      # flip q1 if q0 is 1
        else:
            getattr(qc, g.lower())(q)        # qc.x(q), qc.h(q), ...
    return qc

def quantum_filter(img, gates, shots=1024, width=192):
    orig_size = img.size
    h = max(2, round(width * img.height / img.width))     # keep aspect ratio
    img = img.convert("RGB").resize((width, h))
    px = np.asarray(img, dtype=float) / 255.0        # brightness 0-1, per RGB channel
    b0 = px[:, 0::2, :].ravel()                      # left pixel of each pair -> q0
    b1 = px[:, 1::2, :].ravel()                      # right pixel of each pair -> q1

    # 1) Encode: brightness b -> RY rotation, so P(measure 1) = b
    t0, t1 = 2*np.arcsin(np.sqrt(b0)), 2*np.arcsin(np.sqrt(b1))
    s0 = np.stack([np.cos(t0/2), np.sin(t0/2)], -1)
    s1 = np.stack([np.cos(t1/2), np.sin(t1/2)], -1)
    psi = (s1[:, :, None] * s0[:, None, :]).reshape(-1, 4)   # 2-qubit state |q1 q0>

    # 2) Apply the user's gates (Qiskit circuit -> unitary matrix)
    U = Operator(build_circuit(gates)).data
    probs = np.abs(psi @ U.T) ** 2

    # 3) Measure: estimate P(qubit = 1) by sampling `shots` times
    p0 = np.clip(probs[:, 1] + probs[:, 3], 0, 1)
    p1 = np.clip(probs[:, 2] + probs[:, 3], 0, 1)
    p0 = np.random.binomial(shots, p0) / shots
    p1 = np.random.binomial(shots, p1) / shots

    # 4) Decode: probabilities back into pixel brightness, scaled back up
    out = np.empty_like(px)
    out[:, 0::2, :] = p0.reshape(h, width // 2, 3)
    out[:, 1::2, :] = p1.reshape(h, width // 2, 3)
    result = Image.fromarray((out * 255).astype(np.uint8))
    return result.resize(orig_size, Image.NEAREST)

In [ ]:
import gradio as gr

# ---------- Figma palette ----------
CREAM, BEIGE, PANEL = "#F8F1E5", "#E9E4D2", "#D3CCBF"
INK, ACCENT = "#242424", "#7A4A45"

def draw(gates):
    """Draw the circuit as HTML with vertical wires, like the Figma design."""
    steps = ""
    for g, q in gates:
        if g == "CNOT":
            steps += ('<div class="step cnot"><div class="cell"><span class="ctrl"></span></div>'
                      '<div class="cell"><span class="tgt">+</span></div></div>')
        else:
            c0 = f'<span class="chip">{g}</span>' if q == 0 else ""
            c1 = f'<span class="chip">{g}</span>' if q == 1 else ""
            steps += f'<div class="step"><div class="cell">{c0}</div><div class="cell">{c1}</div></div>'
    if not gates:
        steps = '<div class="empty">Click a gate<br>to add it to a wire</div>'
    return (f'<div class="circ"><div class="head"><span>q<sub>0</sub></span><span>q<sub>1</sub></span></div>'
            f'<div class="wires">{steps}</div></div>')

def add_gate(gates, gate, qubit):
    gates = gates + [(gate, 0 if qubit == "q₀" else 1)]
    return gates, draw(gates)

def clear():
    return [], draw([])

def run(img, gates, shots):
    if img is None:
        raise gr.Error("Upload an image first.")
    return quantum_filter(img, gates, int(shots))

theme = gr.themes.Base(
    font=[gr.themes.GoogleFont("IBM Plex Sans Condensed"), "sans-serif"],
    font_mono=[gr.themes.GoogleFont("IBM Plex Mono"), "monospace"],
).set(
    body_background_fill=CREAM, body_text_color=INK,
    panel_background_fill=PANEL, panel_border_width="0px",
    block_background_fill=BEIGE, block_border_width="0px", block_radius="12px",
    block_label_background_fill=BEIGE, block_label_text_color=INK, block_title_text_color=INK,
    button_secondary_background_fill=CREAM, button_secondary_background_fill_hover="#FFFFFF",
    button_secondary_text_color=INK, button_secondary_border_color=CREAM,
    button_primary_background_fill=ACCENT, button_primary_background_fill_hover="#5E3833",
    button_primary_text_color="#FFFFFF",
    slider_color=ACCENT, checkbox_background_color_selected=ACCENT,
)

css = f"""
@import url('https://fonts.googleapis.com/css2?family=Rubik:wght@800&family=Caveat:wght@600&display=swap');
footer {{display: none !important;}}
body {{background-color: {CREAM} !important;
       background-image: linear-gradient(to top right,
                             {CREAM}D9 0%, {CREAM}C0 22%, {CREAM}00 50%, {CREAM}C0 78%, {CREAM}D9 100%),
                         radial-gradient({ACCENT} 2px, transparent 2.3px) !important;
       background-size: 100% 100%, 34px 34px !important;
       background-attachment: fixed !important;}}
gradio-app, .gradio-container {{background: transparent !important;}}
.gradio-container {{max-width: 1560px !important; width: 100% !important; margin: auto !important;
          padding: 0 24px !important;}}
.gradio-container .main, .gradio-container .wrap, .gradio-container .contain {{max-width: 100% !important;}}
#topbar {{background: {PANEL}; border-radius: 12px; padding: 8px 12px 11px; text-align: center;
          font-family: 'Rubik', sans-serif; font-weight: 800; font-size: 24px; letter-spacing: .02em;
          color: {CREAM};
          text-shadow: 1px 1px 0 {ACCENT}, 2px 2px 0 {ACCENT}, 3px 3px 0 #5E3833, 4px 4px 6px rgba(0,0,0,.18);}}
#sub {{font-family: 'Caveat', cursive; font-size: 22px; font-weight: 600;
       letter-spacing: 0; color: {ACCENT}; text-shadow: none; margin-top: 2px;}}
#cap-gates, #cap-circuit, #cap-image {{text-align: center; letter-spacing: .15em; font-size: 13px;
          padding-bottom: 8px; border-bottom: 2px solid {BEIGE};}}
#gx, #gy, #gz, #gh, #gcnot {{font-family: var(--font-mono) !important; font-size: 20px !important;
          font-weight: 700 !important; border-radius: 10px !important; min-height: 48px;}}
#clear {{background: transparent !important; border: 1.5px solid {ACCENT} !important; color: {ACCENT} !important;}}
#hint {{font-size: 13px; color: #5E665C; background: {CREAM}; border-radius: 10px; padding: 10px;}}
#target {{text-align: center;}}
#target [data-testid="block-info"], #target legend {{display: block !important; width: 100% !important;
          text-align: center !important; margin-bottom: 10px !important;}}
#target .wrap {{display: grid !important; grid-template-columns: 1fr 1fr; gap: 10px !important;}}
#target .wrap label {{justify-content: center !important; min-height: 56px; border-radius: 10px !important;
          font-family: var(--font-mono) !important; font-size: 20px !important; font-weight: 700 !important;}}
#imgrow {{flex-wrap: nowrap !important; gap: 12px;}}
#imgrow > div {{min-width: 0 !important; flex: 1 1 0 !important;}}

/* circuit */
.circ {{font-family: var(--font-mono);}}
.circ .head {{display: flex;}}
.circ .head span {{flex: 1; text-align: center; font-size: 22px; font-weight: 700;}}
.circ .wires {{position: relative; min-height: 340px; padding: 12px 0;
          display: flex; flex-direction: column; gap: 14px;}}
.circ .wires::before, .circ .wires::after {{content: ""; position: absolute; top: 0; bottom: 0;
          width: 2px; margin-left: -1px; background: {INK}; opacity: .6;}}
.circ .wires::before {{left: 25%;}}
.circ .wires::after {{left: 75%;}}
.circ .step {{position: relative; display: flex; z-index: 1;}}
.circ .cell {{flex: 1; display: flex; justify-content: center; align-items: center; min-height: 40px;}}
.circ .chip {{background: {CREAM}; border-radius: 8px; width: 40px; height: 40px; display: flex;
          align-items: center; justify-content: center; font-size: 18px; font-weight: 700;}}
.circ .cnot::before {{content: ""; position: absolute; left: 25%; right: 25%; top: 50%;
          height: 2px; background: {ACCENT};}}
.circ .ctrl {{width: 14px; height: 14px; border-radius: 50%; background: {ACCENT}; z-index: 1;}}
.circ .tgt {{width: 32px; height: 32px; border-radius: 50%; border: 2px solid {ACCENT}; background: {CREAM};
          color: {ACCENT}; display: flex; align-items: center; justify-content: center;
          font-size: 22px; font-weight: 700; z-index: 1;}}
.circ .empty {{position: relative; z-index: 1; margin: 110px 12px 0; padding: 14px 8px; text-align: center;
          font-size: 13px; color: {ACCENT}; background: {PANEL};
          border: 1.5px dashed {ACCENT}; border-radius: 10px;}}
"""

force_light = """
function() {
  const url = new URL(window.location);
  if (url.searchParams.get('__theme') !== 'light') {
    url.searchParams.set('__theme', 'light');
    window.location.href = url.href;
  }
}
"""

with gr.Blocks(theme=theme, css=css, js=force_light, fill_width=True) as demo:
    gates = gr.State([])
    gr.HTML('<div id="topbar">QuantImg<div id="sub">Learn quantum gates by seeing them, one pixel at a time.</div></div>')
    with gr.Row(equal_height=True):
        # GATES panel
        with gr.Column(scale=2, min_width=220, variant="panel"):
            gr.HTML('<div id="cap-gates">GATES</div>')
            target = gr.Radio(["q₀", "q₁"], value="q₀", label="Target qubit", elem_id="target")
            with gr.Row():
                bx = gr.Button("X", elem_id="gx", min_width=60)
                by = gr.Button("Y", elem_id="gy", min_width=60)
            with gr.Row():
                bz = gr.Button("Z", elem_id="gz", min_width=60)
                bh = gr.Button("H", elem_id="gh", min_width=60)
            bc = gr.Button("CNOT", elem_id="gcnot")
            gr.HTML('<div id="hint"><b>X</b> inverts · <b>H</b> mixes toward gray · '
                    '<b>CNOT</b> links neighbouring pixels</div>')
        # CIRCUIT panel
        with gr.Column(scale=3, min_width=300, variant="panel"):
            gr.HTML('<div id="cap-circuit">CIRCUIT</div>')
            circ = gr.HTML(draw([]))
            shots = gr.Slider(10, 2000, value=1024, step=10, label="Shots (fewer = noisier)")
            clr = gr.Button("Clear", elem_id="clear")
            run_b = gr.Button("▶ Run", variant="primary", size="lg")
        # IMAGES
        with gr.Column(scale=9, min_width=640, variant="panel"):
            gr.HTML('<div id="cap-image">IMAGE</div>')
            with gr.Row(elem_id="imgrow"):
                try:
                    inp = gr.Image(type="pil", label="Before", height=520,
                                   placeholder="Upload image\n\nor drag it here")
                except TypeError:
                    inp = gr.Image(type="pil", label="Before", height=520)
                out = gr.Image(label="After", height=520)

    for b, g in [(bx, "X"), (by, "Y"), (bz, "Z"), (bh, "H"), (bc, "CNOT")]:
        b.click(lambda gs, q, g=g: add_gate(gs, g, q), [gates, target], [gates, circ])
    clr.click(clear, None, [gates, circ])
    run_b.click(run, [inp, gates, shots], out)

demo.launch(share=True, debug=True)

/tmp/ipykernel_49048/2960686417.py:119: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css, js. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=theme, css=css, js=force_light, fill_width=True) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://776f6457df2a2a94cc.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
